# 4. Multi-Tenant Vector Databasing with Milvus for RAG and Fine-tuning with GPU/QAIC (Instructor: Mohammad Sada)

The tutorial focuses on NRP-managed Milvus as a multi-tenant vector database service. Participants are guided through accessing Milvus with their credentials, defining collections, and inserting embeddings for semantic search. Using either GPU or QAIC resources, attendees will integrate Milvus into RAG or fine-tuning workflows.

## Milvus on NRP

NRP runs **Milvus** as a managed service, so you don’t install the operator or create the instance yourself. The gRPC endpoint is **`milvus.nrp-nautilus.io:50051`**. To get access, use the [Milvus password page](https://nrp.ai/milvus) and click “Get milvus password”; the secure link is emailed to you. You must be in a [namespace/group with Milvus enabled](https://nrp.ai/namespaces). Group names may use letters, numbers, and dashes; any dashes are converted to underscores in the Milvus database name. From there you can [define collections](https://milvus.io/docs/manage-collections.md), run [vector search](https://milvus.io/docs/single-vector-search.md), or use the [Attu GUI](https://milvus.io/docs/quickstart_with_attu.md). Full reference: [NRP vector database](https://nrp.ai/documentation/userdocs/vector-database/).

## RAG example using Milvus
### Credit: Mohammad Sada, SDSC

**Prerequisite:** Get your Milvus password and ensure your group has Milvus enabled (see “Milvus on NRP” above) before starting the pod.

This example demonstrates RAG using Milvus as the vector database. In `yamls/milvus-rag.yaml`, **replace `<username>`** with your username so the pod name `tutorial-<username>-vectordb` is unique. Start up the pod (namespace `nrp-training-k8s`; create it with `kubectl create namespace nrp-training-k8s` if needed):
```
kubectl apply -n nrp-training-k8s -f yamls/milvus-rag.yaml
```

In [ ]:
# Run this cell once to apply the NRP Tutorial theme (optional)
from IPython.display import HTML, display
display(HTML("""
<style>
  /* NRP Tutorial theme — indigo / slate / teal */
  .jp-MarkdownCell .jp-Cell-inputWrapper,
  .text_cell .input_area {
    background: linear-gradient(90deg, #eef2ff 0%, #f5f3ff 10%, #fafafa 22%);
    border-left: 4px solid #6366f1;
    padding: 0.75em 0 0.75em 1em;
    margin: 0.5em 0;
    border-radius: 0 8px 8px 0;
    box-shadow: 0 1px 3px rgba(99, 102, 241, 0.08);
  }
  .jp-MarkdownCell h1, .text_cell_render h1 {
    color: #312e81;
    font-weight: 700;
    border-bottom: 2px solid #6366f1;
    padding-bottom: 0.45em;
    margin-top: 0;
    background: linear-gradient(90deg, #e0e7ff 0%, rgba(224, 231, 255, 0.4) 70%, transparent 100%);
    padding: 0.35em 0 0.45em 0.6em;
    margin: 0 -0.6em 0.35em -0.6em;
    border-radius: 6px 6px 0 0;
  }
  .jp-MarkdownCell h2, .text_cell_render h2 {
    color: #4338ca;
    font-weight: 600;
    margin-top: 1.25em;
    margin-bottom: 0.45em;
    padding-left: 0.5em;
    border-left: 4px solid #6366f1;
    background: linear-gradient(90deg, rgba(99, 102, 241, 0.06) 0%, transparent 100%);
    padding: 0.25em 0.5em 0.25em 0.6em;
    border-radius: 0 4px 4px 0;
  }
  .jp-MarkdownCell h3, .text_cell_render h3 {
    color: #0f766e;
    font-weight: 600;
    margin-top: 1em;
    padding-left: 0.45em;
    border-left: 4px solid #14b8a6;
    background: linear-gradient(90deg, rgba(20, 184, 166, 0.06) 0%, transparent 100%);
    padding: 0.2em 0.45em 0.2em 0.55em;
    border-radius: 0 4px 4px 0;
  }
  .jp-MarkdownCell p, .jp-MarkdownCell li, .text_cell_render p, .text_cell_render li {
    line-height: 1.7;
    color: #374151;
  }
  .jp-MarkdownCell a, .text_cell_render a {
    color: #4f46e5;
    font-weight: 500;
  }
  .jp-MarkdownCell a:hover, .text_cell_render a:hover {
    color: #4338ca;
    text-decoration: underline;
  }
  .jp-MarkdownCell code, .text_cell_render code {
    background: #e0e7ff;
    color: #3730a3;
    padding: 0.22em 0.5em;
    border-radius: 5px;
    font-size: 0.9em;
    border: 1px solid #c7d2fe;
  }
  .jp-MarkdownCell pre, .text_cell_render pre {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-left: 4px solid #6366f1;
    border-radius: 6px;
    padding: 0.8em 1em;
  }
  .jp-MarkdownCell strong, .text_cell_render strong {
    color: #1e293b;
  }
  .jp-MarkdownCell hr, .text_cell_render hr {
    border: none;
    border-top: 2px solid #cbd5e1;
    margin: 1.5em 0;
  }
  .jp-MarkdownCell ul, .text_cell_render ul {
    padding-left: 1.5em;
  }
  .jp-MarkdownCell li::marker, .text_cell_render li::marker {
    color: #6366f1;
  }
  .jp-MarkdownCell table, .text_cell_render table {
    border-collapse: collapse;
    border: 1px solid #e2e8f0;
    border-radius: 6px;
    overflow: hidden;
  }
  .jp-MarkdownCell th, .text_cell_render th {
    background: #eef2ff;
    color: #3730a3;
    padding: 0.5em 0.75em;
    text-align: left;
  }
  .jp-MarkdownCell td, .text_cell_render td {
    padding: 0.45em 0.75em;
    border-top: 1px solid #e2e8f0;
  }
</style>
"""))

Apply the Milvus RAG pod:

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/milvus-rag.yaml

Watch the logs and make sure you wait till the installs are done:

```
kubectl logs -n nrp-training-k8s tutorial-<username>-vectordb
```
Replace `<username>` with the name you used in the YAML.

In [ ]:
!kubectl logs -n nrp-training-k8s tutorial-<username>-vectordb

The pod automatically installs Python dependencies, Ollama, starts the Ollama server, and downloads the mistral model. Download the example script into the pod once it is running (example repo: [groundsada/nrp-milvus-example](https://github.com/groundsada/nrp-milvus-example)):
```
kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# then inside the pod:
cd /scratch
wget https://raw.githubusercontent.com/groundsada/nrp-milvus-example/refs/heads/main/milvus-example.py
```

In [ ]:
!kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# In the next cell or inside the pod: cd /scratch && wget https://raw.githubusercontent.com/groundsada/nrp-milvus-example/refs/heads/main/milvus-example.py

Once the installation is complete (check logs), you can run the example. This example connects to the `sc25_milvus` database using credentials from the Kubernetes secret `sc25-milvus-credentials`. The collection name is `simple_rag_example`. To use a different collection, edit `milvus-example.py` and change `collection_name="simple_rag_example"` (there are two places where the collection name is set).

Run the script inside the pod (see next cell). The script uses environment variables for the Milvus connection.

For all other aspects, the script uses environment variables for Milvus connection, so no manual editing is needed.

```
kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# then inside the pod:
cd /scratch
python3 milvus-example.py
```

In [ ]:
!kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# Then run: cd /scratch && python3 milvus-example.py

---
## Embedding Generation: NVIDIA vs Qualcomm Backends

The embedding and RAG workflow above uses an **NVIDIA GPU** pod with Ollama. The same workflow can be adapted for **Qualcomm Cloud AI 100 Ultra**:

| Component | NVIDIA GPU | Qualcomm Cloud AI 100 |
|-----------|-----------|----------------------|
| **Resource request** | `nvidia.com/gpu: 1` | `qualcomm.com/qaic: 1` |
| **Container image** | `ollama/ollama` (or custom) | `ghcr.io/quic/cloud_ai_inference_ubuntu24:1.20.6.0` |
| **Inference engine** | Ollama / vLLM (CUDA) | vLLM (QAIC backend) |
| **Embedding model** | `mistral` via Ollama | Models compiled for QAIC via `/opt/vllm-env/` |

To run the Milvus RAG pipeline with a Qualcomm backend: (1) Deploy a Qualcomm inference pod using `yamls/qaic-inference.yaml` (Part 3). (2) Serve an embedding model via the vLLM OpenAI-compatible API on the QAIC device. (3) Point your embedding client at the Qualcomm pod's service endpoint. Milvus storage and retrieval stay the same. See the [Cloud AI SDK Kubernetes docs](https://quic.github.io/cloud-ai-sdk-pages/latest/Getting-Started/Installation/vLLM/vLLM/index.html) for Qualcomm deployment details.


This example uses a small set of sample documents to demonstrate Milvus vector storage and retrieval and RAG with the Ollama LLM.

## End

**Please make sure you did not leave any running pods. Jobs and associated completed pods are OK.**

**Join NRP & hands-on support:** Use these links to [get started with NRP](https://nrp.ai/documentation/userdocs/start/getting-started/) and to [join NRP's Matrix chat](https://nrp.ai/contact/) for hands-on support during the tutorial.

**Related documentation:** [NRP-managed vector database (Milvus)](https://nrp.ai/documentation/userdocs/vector-database/) · [NRP-managed LLMs](https://nrp.ai/documentation/userdocs/ai/llm-managed/) · [Qualcomm Cloud AI 100](https://nrp.ai/documentation/userdocs/qaic/)